# Social Media Content Engagement Analysis

DA-020 Digital Marketing | Python, Pandas and Matplotlib

This notebook analyzes synthetic post-level exports to compare post formats, publishing times and dates.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Works when launched from the repository root or the notebooks folder
csv_path = Path('data/social_media_posts.csv')
if not csv_path.exists():
    csv_path = Path('../data/social_media_posts.csv')
df = pd.read_csv(csv_path)
df.head()


## 1. Data cleaning
Missing numeric metrics are filled with zero for this demonstration. In a real campaign, validate whether a blank means zero, unavailable, or not yet reported.

In [ ]:
metric_cols = ['impressions','reach','likes','comments','shares']
df[metric_cols] = df[metric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
df['date'] = pd.to_datetime(df['date'])
df['total_engagements'] = df[['likes','comments','shares']].sum(axis=1)
df['engagement_rate'] = np.where(df['reach'] > 0, df['total_engagements'] / df['reach'] * 100, 0)
df['impressions_to_reach'] = np.where(df['reach'] > 0, df['impressions'] / df['reach'], 0)
df.head()

## 2. Metric definitions
- Impressions: total displays of a post.
- Reach: unique accounts that saw a post.
- Total engagements: likes + comments + shares.
- Engagement rate by reach: total engagements / reach × 100.
- Impressions-to-reach ratio: impressions / reach.

Reach is used as the denominator because it estimates interactions relative to the unique audience exposed. Do not compare this directly with a rate using impressions as denominator.

In [ ]:
summary = df.groupby('post_type').agg(posts=('post_id','count'), avg_reach=('reach','mean'), avg_engagement_rate=('engagement_rate','mean'), total_engagements=('total_engagements','sum')).sort_values('avg_engagement_rate', ascending=False)
summary

## 3. Compare publishing times and dates

In [ ]:
time_summary = df.groupby('posting_time')['engagement_rate'].mean().reindex(['08:00','12:00','17:00','20:00'])
print(time_summary)
daily_summary = df.groupby('date')['engagement_rate'].mean()
daily_summary.describe()

## 4. Four engagement charts

In [ ]:
df.groupby('post_type')['engagement_rate'].mean().sort_values().plot(kind='bar', title='Average Engagement Rate by Post Type')
plt.ylabel('Engagement rate (%)'); plt.tight_layout(); plt.show()

df.groupby('posting_time')['engagement_rate'].mean().reindex(['08:00','12:00','17:00','20:00']).plot(kind='bar', title='Average Engagement Rate by Posting Time')
plt.ylabel('Engagement rate (%)'); plt.tight_layout(); plt.show()

df.groupby('date')['engagement_rate'].mean().plot(marker='o', title='Daily Average Engagement Rate')
plt.ylabel('Engagement rate (%)'); plt.tight_layout(); plt.show()

df.nlargest(10,'engagement_rate').sort_values('engagement_rate').plot(x='post_id', y='engagement_rate', kind='barh', legend=False, title='Top 10 Posts by Engagement Rate')
plt.xlabel('Engagement rate (%)'); plt.tight_layout(); plt.show()

## 5. High-performing posts

In [ ]:
ranking = df.sort_values('engagement_rate', ascending=False)[['post_id','date','posting_time','platform','post_type','reach','total_engagements','engagement_rate']]
ranking.head(10)

## 6. Content experiments
1. Format test: compare Reels and carousels on similar topics and consistent creative quality.
2. Time test: compare midday and evening slots, rotating weekdays to reduce day-of-week bias.
3. CTA test: compare posts with a question or save/share prompt against posts without one.
4. Repeat each test across multiple posts; compare median engagement rate, reach, shares and comments.

**Limitations:** synthetic data is for academic demonstration, not real campaign results. Observational associations do not prove causation; platform metric definitions may differ.